<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/AeroPINNN_X_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# AeroPINN_X_v3_Colab.py
# Final stage-runner notebook script
# ==================================

from __future__ import annotations

import os
import gc
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from airfoil import AirfoilGeometry, AirfoilFormatError
from parsec_fit import (
    fit_parsec_to_dat,
    fit_quality_report,
    preprocess_airfoil_for_parsec,
)
from network import MLP, load_checkpoint, save_checkpoint
from train_loop import (
    train_multi_airfoil,
    optimize_shape,
)
from postprocess import (
    validate_model_on_library,
    export_optimized_airfoil,
    export_run_report,
)
from xai import run_xai_suite

In [2]:
# ============================================================================
# CELL 1 ─ Environment
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/MyDrive')

In [3]:
# ============================================================================
# CELL 2 ─ User stage selection
# ============================================================================
# STAGE = 1 -> pilot
# STAGE = 2 -> research training
# STAGE = 3 -> optimize one selected airfoil using trained model
STAGE = 2

# folders
DATA_ROOT = Path("/content/airfoils")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")
CKPT_DIR = OUT_DIR / "checkpoints"
VAL_OUT_DIR = OUT_DIR / "validation"
OPT_OUT_DIR = OUT_DIR / "optimized_results"
XAI_OUT_DIR = OUT_DIR / "xai_outputs"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
OPT_OUT_DIR.mkdir(parents=True, exist_ok=True)
XAI_OUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = None   # e.g. CKPT_DIR / "multi_airfoil_last.pt"



In [4]:
# ============================================================================
# CELL 3 ─ Config
# ============================================================================
if STAGE == 1:
    AOA_LIST = [-2, 0, 2, 4]
    RE_LIST = [5e4, 1e5]

    N_INT = 2000
    N_NEAR = 400
    N_WAKE = 400
    N_AIRFOIL = 600
    N_INLET = 400
    N_OUTLET = 400
    N_TOP = 400
    N_BOT = 400

    ADAM_STEPS = 1500
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 300
    ADAPTIVE_CAP = 1000
    CANDIDATE_POOL = 4000
    GAUSS_PER_CENTER = 4
    RESIDUAL_PROBE_STRIDE = 6

elif STAGE == 2:
    AOA_LIST = [0.0, 4.0]
    RE_LIST = [1e6, 2e6]

    N_INT = 3000
    N_NEAR = 600
    N_WAKE = 600
    N_AIRFOIL = 800
    N_INLET = 500
    N_OUTLET = 500
    N_TOP = 500
    N_BOT = 500

    ADAM_STEPS = 5000
    LBFGS_STEPS = 10
    ADAPTIVE_EVERY = 100
    ADAPTIVE_CAP = 2000
    CANDIDATE_POOL = 6000
    GAUSS_PER_CENTER = 6
    RESIDUAL_PROBE_STRIDE = 4

else:  # STAGE 3
    AOA_LIST = [5.0]
    RE_LIST = [1e5]

    # Stage 3 is optimization, not large retraining
    N_INT = 1500
    N_NEAR = 300
    N_WAKE = 300
    N_AIRFOIL = 400
    N_INLET = 250
    N_OUTLET = 250
    N_TOP = 250
    N_BOT = 250

    ADAM_STEPS = 0
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 0
    ADAPTIVE_CAP = 0
    CANDIDATE_POOL = 0
    GAUSS_PER_CENTER = 0
    RESIDUAL_PROBE_STRIDE = 1

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

PARSEC_OPT_THRESHOLD = 4.5

In [5]:
# ============================================================================
# CELL 4 ─ Build train/val/test airfoil library
# ============================================================================
def load_airfoil_library(folder: Path, n_boundary: int = 1200):
    lib = []
    skipped = []

    for f in sorted(folder.glob("*.dat")):
        try:
            geom = AirfoilGeometry.from_dat(f, n_boundary=n_boundary)

            parsec_params = None
            fit_report = None
            fit_mse = None
            use_for_optimization = False

            try:
                pre = preprocess_airfoil_for_parsec(geom.boundary, n_surface=300)
                params, fit_mse = fit_parsec_to_dat(str(f), verbose=False)
                fit_report = fit_quality_report(pre["upper"], pre["lower"], params)

                if fit_report["max_error_pct"] <= PARSEC_OPT_THRESHOLD:
                    parsec_params = params
                    use_for_optimization = True

            except Exception as e_fit:
                fit_report = {
                    "quality": "POOR",
                    "max_error_pct": 100.0,
                    "mean_error_pct": 100.0,
                    "error_message": str(e_fit),
                }

            lib.append(
                {
                    "name": f.stem,
                    "path": str(f),
                    "geometry": geom,
                    "parsec_params": parsec_params,
                    "fit_report": fit_report,
                    "fit_mse": fit_mse,
                    "use_for_optimization": use_for_optimization,
                }
            )

        except Exception as e:
            skipped.append((f.name, str(e)))

    return lib, skipped


TRAIN_AIRFOILS, TRAIN_SKIPPED = load_airfoil_library(TRAIN_DIR)
VAL_AIRFOILS, VAL_SKIPPED = load_airfoil_library(VAL_DIR)
TEST_AIRFOILS, TEST_SKIPPED = load_airfoil_library(TEST_DIR)

print(f"Loaded {len(TRAIN_AIRFOILS)} training airfoils")
print(f"Loaded {len(VAL_AIRFOILS)} validation airfoils")
print(f"Loaded {len(TEST_AIRFOILS)} test airfoils")

if TRAIN_SKIPPED or VAL_SKIPPED or TEST_SKIPPED:
    print("\nSkipped files:")
    for item in TRAIN_SKIPPED + VAL_SKIPPED + TEST_SKIPPED:
        print(" ", item)

Loaded 28 training airfoils
Loaded 4 validation airfoils
Loaded 4 test airfoils


In [6]:
# ============================================================================
# DEVICE SETUP
# ============================================================================
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [7]:
# ============================================================================
# CELL 5 ─ Model create or resume
# ============================================================================
model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

# if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
#     model = load_checkpoint(str(RESUME_CHECKPOINT), device=device)
#     print("Resumed model from:", RESUME_CHECKPOINT)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))



MLP(5→4, 64×16, act=silu, params=64,068)
Total parameters: 64068


In [ ]:
# # ============================================================================
# # CELL 6 ─ Train stages 1/2
# # ============================================================================
# history = []
# adaptive_weights = None

# if STAGE in (1, 2):
#     print("=" * 62)
#     print(" STAGE 1: Pilot" if STAGE == 1 else " STAGE 2: Research")
#     print(f" AdamW steps = {ADAM_STEPS}")
#     print(f" L-BFGS steps = {LBFGS_STEPS}")
#     print(f" Adaptive every = {ADAPTIVE_EVERY}")
#     print(f" Adaptive cap   = {ADAPTIVE_CAP}")
#     print("=" * 62)

#     model, adaptive_weights, history = train_multi_airfoil(
#         model=model,
#         train_airfoils=TRAIN_AIRFOILS,
#         val_airfoils=VAL_AIRFOILS,
#         aoa_list=AOA_LIST,
#         re_list=RE_LIST,
#         xlim=XLIM,
#         ylim=YLIM,
#         N_int=N_INT,
#         N_near=N_NEAR,
#         N_airfoil=N_AIRFOIL,
#         N_inlet=N_INLET,
#         N_outlet=N_OUTLET,
#         N_top=N_TOP,
#         N_bot=N_BOT,
#         N_wake=N_WAKE,
#         near_band=NEAR_BAND,
#         adam_steps=ADAM_STEPS,
#         lr_adam=LR_ADAM,
#         lr_min=LR_MIN,
#         lr_lambda=LR_LAMBDA,
#         weight_decay=WEIGHT_DECAY,
#         refresh_every=ADAPTIVE_EVERY,
#         adaptive_every=ADAPTIVE_EVERY,
#         adaptive_cap=ADAPTIVE_CAP,
#         candidate_pool=CANDIDATE_POOL,
#         gauss_per_center=GAUSS_PER_CENTER,
#         residual_probe_stride=RESIDUAL_PROBE_STRIDE,
#         lbfgs_steps=LBFGS_STEPS,
#         print_every=PRINT_EVERY,
#         save_every=SAVE_EVERY,
#         val_every=VAL_EVERY,
#         out_dir=str(CKPT_DIR),
#         device=device,
#         seed=42,
#         resume_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
#     )

#     print("Training finished.")
#     if adaptive_weights is not None:
#         print("Final adaptive weights:", adaptive_weights.all_weights())

#     save_checkpoint(model, str(CKPT_DIR / "final_model.pt"))


In [ ]:
import importlib
import train_loop


importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [ ]:
# ============================================================================
# HIGH-Re 100-COMBINATION CONFIG
# ============================================================================

STAGE = 2

# 25 airfoils × (2 AoA × 2 Re) = 100 combinations
AOA_LIST = [0.0, 4.0]
RE_LIST  = [1.0e6, 2.0e6]

# research-lite but high-Re safe config
N_INT = 3000
N_NEAR = 600
N_WAKE = 600
N_AIRFOIL = 800
N_INLET = 500
N_OUTLET = 500
N_TOP = 500
N_BOT = 500

ADAM_STEPS = 5000
LBFGS_STEPS = 10

ADAPTIVE_EVERY = 100
ADAPTIVE_CAP = 2000
CANDIDATE_POOL = 6000
GAUSS_PER_CENTER = 6
RESIDUAL_PROBE_STRIDE = 4

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

In [ ]:
import importlib
import train_loop
importlib.reload(train_loop)

from train_loop import train_multi_airfoil
print(train_multi_airfoil)

<function train_multi_airfoil at 0x7e3ecc7c36a0>


In [ ]:
import importlib
import losses
import train_loop

importlib.reload(losses)
importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [ ]:
from pathlib import Path
import json

state_file = Path(OUT_DIR) / "combo_state.json"
state = json.loads(state_file.read_text())
print(state)

{'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [ ]:
from pathlib import Path
import json

state_file = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
state = json.loads(state_file.read_text())

state["next_block_index"] = 90
state["finished_all_selected_combos"] = False

state_file.write_text(json.dumps(state, indent=2))
print("combo_state.json fixed for resume from human block 91")
print(state)

combo_state.json fixed for resume from human block 91
{'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
from pathlib import Path
import json
import torch

STATE_FILE = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
RESUME_CHECKPOINT = Path("/content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt")
OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")

print("Checkpoint exists:", RESUME_CHECKPOINT.exists())
print("State exists:", STATE_FILE.exists())

state = json.loads(STATE_FILE.read_text())
state["next_block_index"] = 90
state["finished_all_selected_combos"] = False
STATE_FILE.write_text(json.dumps(state, indent=2))
print("Updated state:", state)

ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
print("Checkpoint keys:", ckpt.keys())

model.load_state_dict(ckpt["model_state"])
print("Loaded model_state from:", RESUME_CHECKPOINT)

history = ckpt.get("history", {})
print("Saved step:", ckpt.get("step"))
print("Saved block_index:", ckpt.get("block_index"))

Checkpoint exists: True
State exists: True
Updated state: {'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}
Checkpoint keys: dict_keys(['step', 'block_index', 'model_state', 'optimizer_state', 'scheduler_state', 'lambda_optimizer_state', 'loss_weights_state', 'adaptive_sampler_state', 'history', 'extra'])
Loaded model_state from: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
Saved step: 5000
Saved block_index: 90


In [ ]:
# ============================================================================
# CELL ─ High-Re run settings
# ============================================================================

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

print("BLOCK_STEPS =", BLOCK_STEPS)
print("BLOCKS_PER_RUN =", BLOCKS_PER_RUN)
print("MAX_TOTAL_COMBOS =", MAX_TOTAL_COMBOS)

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100


In [ ]:
# Check which required variables are still missing
required_names = [
    "TRAIN_AIRFOILS", "VAL_AIRFOILS",
    "AOA_LIST", "RE_LIST",
    "XLIM", "YLIM",
    "N_INT", "N_NEAR", "N_AIRFOIL", "N_INLET", "N_OUTLET", "N_TOP", "N_BOT", "N_WAKE",
    "NEAR_BAND",
    "ADAM_STEPS", "LBFGS_STEPS",
    "LR_ADAM", "LR_MIN", "LR_LAMBDA", "WEIGHT_DECAY",
    "ADAPTIVE_EVERY", "ADAPTIVE_CAP",
    "CANDIDATE_POOL", "GAUSS_PER_CENTER", "RESIDUAL_PROBE_STRIDE",
    "PRINT_EVERY", "SAVE_EVERY", "VAL_EVERY",
    "train_multi_airfoil"
]

missing = [name for name in required_names if name not in globals()]
print("Missing:", missing)

Missing: []


In [ ]:
# ============================================================================
# CELL ─ Run 10 new high-Re combinations
# ============================================================================

print("=" * 62)
print(" STAGE 2: High-Re 100-combination training")
print(f" AdamW steps = {ADAM_STEPS}")
print(f" L-BFGS steps = {LBFGS_STEPS}")
print(f" Adaptive every = {ADAPTIVE_EVERY}")
print(f" Adaptive cap   = {ADAPTIVE_CAP}")
print(f" Blocks/run     = {10}")
print(f" Max combos     = {100}")
print(f" AoA list       = {AOA_LIST}")
print(f" Re list        = {RE_LIST}")
print("=" * 62)

model, adaptive_weights, history = train_multi_airfoil(
    model=model,
    train_airfoils=TRAIN_AIRFOILS,
    val_airfoils=VAL_AIRFOILS,
    aoa_list=[0.0, 4.0],
    re_list=[1e6, 2e6],
    xlim=XLIM,
    ylim=YLIM,
    N_int=N_INT,
    N_near=N_NEAR,
    N_airfoil=N_AIRFOIL,
    N_inlet=N_INLET,
    N_outlet=N_OUTLET,
    N_top=N_TOP,
    N_bot=N_BOT,
    N_wake=N_WAKE,
    near_band=NEAR_BAND,
    adam_steps=ADAM_STEPS,
    lr_adam=LR_ADAM,
    lr_min=LR_MIN,
    lr_lambda=LR_LAMBDA,
    weight_decay=WEIGHT_DECAY,
    refresh_every=ADAPTIVE_EVERY,
    adaptive_every=ADAPTIVE_EVERY,
    adaptive_cap=ADAPTIVE_CAP,
    candidate_pool=CANDIDATE_POOL,
    gauss_per_center=GAUSS_PER_CENTER,
    residual_probe_stride=RESIDUAL_PROBE_STRIDE,
    lbfgs_steps=LBFGS_STEPS,
    print_every=PRINT_EVERY,
    save_every=SAVE_EVERY,
    val_every=VAL_EVERY,
    out_dir=Path("/content/drive/MyDrive/AeroPINN"),
    device=device,
    seed=42,
    block_steps=500,
    blocks_per_run=10,
    max_total_combos=100,
    resume_checkpoint=str(RESUME_CHECKPOINT),
)

print("\nBatch finished.")
if adaptive_weights is not None:
    print("Final adaptive weights:", adaptive_weights.all_weights())
print("History rows:", len(history))

 STAGE 2: High-Re 100-combination training
 AdamW steps = 5000
 L-BFGS steps = 10
 Adaptive every = 100
 Adaptive cap   = 2000
 Blocks/run     = 10
 Max combos     = 100
 AoA list       = [0.0, 4.0]
 Re list        = [1000000.0, 2000000.0]
train_multi_airfoil: 28 airfoils, 4 conditions, selected combos=100, running blocks 90..99

BLOCK 91/100
  Airfoil : naca4412
  AoA     : 0.0 deg
  Re      : 1.000e+06
  Steps   : 500
  step   100  loss=1.1719e-06  pde=1.2880e-06  wall=5.6693e-09  lr=1.52e-04  λwall=0.41  λpde=0.84
  step   200  loss=1.8803e-06  pde=9.8623e-07  wall=4.1450e-09  lr=7.70e-05  λwall=0.13  λpde=1.88
  step   300  loss=1.6351e-06  pde=7.5386e-07  wall=4.0259e-09  lr=3.90e-05  λwall=0.13  λpde=2.13
  step   400  loss=1.5146e-06  pde=6.9703e-07  wall=3.9732e-09  lr=1.97e-05  λwall=0.13  λpde=2.13
  step   500  loss=1.6220e-06  pde=7.4748e-07  wall=4.0744e-09  lr=1.00e-05  λwall=0.13  λpde=2.13

BLOCK 92/100
  Airfoil : naca0015
  AoA     : 0.0 deg
  Re      : 2.000e+06
  St

In [ ]:
import json
from pathlib import Path
import torch

def save_batch_checkpoint_permanent(
    model,
    adaptive_weights,
    history,
    out_dir,
    next_block_index,
    total_selected_combos,
    tag="batch_checkpoint",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = out_dir / f"{tag}.pt"
    state_path = out_dir / "combo_state.json"
    history_path = out_dir / "history.json"

    payload = {
        "model_state": model.state_dict(),
        "loss_weights_state": adaptive_weights.state_dict() if adaptive_weights is not None else None,
        "block_index": int(next_block_index),
        "history": history,
        "extra": {
            "next_block_index": int(next_block_index),
            "total_selected_combos": int(total_selected_combos),
        },
    }
    torch.save(payload, ckpt_path)

    state_path.write_text(
        json.dumps(
            {
                "next_block_index": int(next_block_index),
                "total_selected_combos": int(total_selected_combos),
                "finished_all_selected_combos": bool(next_block_index >= total_selected_combos),
            },
            indent=2,
        )
    )

    history_path.write_text(json.dumps(history, indent=2))

    print("Saved permanent checkpoint to:", ckpt_path)
    print("Saved combo state to        :", state_path)
    print("Saved history to            :", history_path)

In [ ]:
save_batch_checkpoint_permanent(
    model=model,
    adaptive_weights=adaptive_weights,
    history=history,
    out_dir=OUT_DIR,
    next_block_index=10,          # change this to the real next index
    total_selected_combos=100,
    tag="multi_airfoil_last",
)

NameError: name 'adaptive_weights' is not defined

In [ ]:
import json
from pathlib import Path

state_file = Path(OUT_DIR) / "combo_state.json"
state = json.loads(state_file.read_text())

next_block_index = state["next_block_index"]
total_selected_combos = state["total_selected_combos"]

save_batch_checkpoint_permanent(
    model=model,
    adaptive_weights=adaptive_weights,
    history=history,
    out_dir=OUT_DIR,
    next_block_index=next_block_index,
    total_selected_combos=total_selected_combos,
    tag="multi_airfoil_last",
)

NameError: name 'adaptive_weights' is not defined

In [ ]:
#load it later

# import torch
# from pathlib import Path

# ckpt_path = Path(OUT_DIR) / "multi_airfoil_last.pt"
# data = torch.load(ckpt_path, map_location=device, weights_only=False)

# model.load_state_dict(data["model_state"])

# if adaptive_weights is not None and data.get("loss_weights_state") is not None:
#     adaptive_weights.load_state_dict(data["loss_weights_state"])

# history = data.get("history", [])
# next_block_index = data.get("block_index", 0)

# print("Loaded checkpoint from:", ckpt_path)
# print("Next block index:", next_block_index)
# print("History rows:", len(history))

### **Validation**

In [30]:
# ============================================================================
# CELL 7 ─ Validation + reports
# ============================================================================
from pathlib import Path

if STAGE in (1, 2):
    VAL_STAGE_DIR = Path(VAL_OUT_DIR) / f"stage_{STAGE}"
    VAL_STAGE_DIR.mkdir(parents=True, exist_ok=True)

    all_reports = {}

    # ------------------------------------------------------------
    # Validation library
    # ------------------------------------------------------------
    if len(VAL_AIRFOILS) > 0:
        df_val, val_summary = validate_model_on_library(
            model=model,
            airfoil_library=VAL_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir=VAL_STAGE_DIR / "validation",
            device=device,
            max_cases=24,
        )
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)
    else:
        df_val = None
        val_summary = {"message": "No validation airfoils available"}
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)

    # ------------------------------------------------------------
    # Test library (optional but recommended)
    # ------------------------------------------------------------
    if "TEST_AIRFOILS" in globals() and len(TEST_AIRFOILS) > 0:
        df_test, test_summary = validate_model_on_library(
            model=model,
            airfoil_library=TEST_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir=VAL_STAGE_DIR / "test",
            device=device,
            max_cases=24,
        )
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)
    else:
        test_summary = {"message": "No test airfoils available"}
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)

    # ------------------------------------------------------------
    # Export run report
    # ------------------------------------------------------------
    run_report = export_run_report(
        history=history if "history" in globals() else [],
        extra={
            "stage": STAGE,
            "aoa_list": list(AOA_LIST),
            "re_list": list(RE_LIST),
            "validation_summary": val_summary,
            "test_summary": test_summary,
            "n_train_airfoils": len(TRAIN_AIRFOILS) if "TRAIN_AIRFOILS" in globals() else 0,
            "n_val_airfoils": len(VAL_AIRFOILS) if "VAL_AIRFOILS" in globals() else 0,
            "n_test_airfoils": len(TEST_AIRFOILS) if "TEST_AIRFOILS" in globals() else 0,
        },
        out_dir=Path(OUT_DIR) / "run_report",
    )

    print("Run report:", run_report)

Validation summary: {'n_cases': 16, 'mean_CL': 2.9342714557789365e-07, 'mean_CD': -9.612186903709238e-07, 'mean_LD': 0.02149631219953417, 'csv_path': '/content/drive/MyDrive/AeroPINN/validation/stage_2/validation/validation_metrics.csv'}
Test summary: {'n_cases': 16, 'mean_CL': 1.7729822639508084e-07, 'mean_CD': -8.480651003821228e-07, 'mean_LD': 0.1439771410554338, 'csv_path': '/content/drive/MyDrive/AeroPINN/validation/stage_2/test/validation_metrics.csv'}
Run report: {'n_history_rows': 0, 'extra': {'stage': 2, 'aoa_list': [0.0, 4.0], 're_list': [1000000.0, 2000000.0], 'validation_summary': {'n_cases': 16, 'mean_CL': 2.9342714557789365e-07, 'mean_CD': -9.612186903709238e-07, 'mean_LD': 0.02149631219953417, 'csv_path': '/content/drive/MyDrive/AeroPINN/validation/stage_2/validation/validation_metrics.csv'}, 'test_summary': {'n_cases': 16, 'mean_CL': 1.7729822639508084e-07, 'mean_CD': -8.480651003821228e-07, 'mean_LD': 0.1439771410554338, 'csv_path': '/content/drive/MyDrive/AeroPINN/val

In [31]:
import pandas as pd

df_val = pd.read_csv("/content/drive/MyDrive/AeroPINN/validation/stage_2/validation/validation_metrics.csv")
df_test = pd.read_csv("/content/drive/MyDrive/AeroPINN/validation/stage_2/test/validation_metrics.csv")

print(df_val.head())
print(df_val.columns.tolist())
print(df_test.head())
print(df_test.columns.tolist())

    airfoil  alpha_deg    Re_phys            CL            CD        LD
0  naca0011        0.0  1000000.0  1.206572e-06 -2.779071e-06 -0.434164
1  naca0011        0.0  2000000.0  1.139469e-06 -3.042655e-06 -0.374498
2  naca0011        4.0  1000000.0  1.429059e-06 -2.580095e-06 -0.553878
3  naca0011        4.0  2000000.0  1.381140e-06 -2.844046e-06 -0.485625
4  naca2414        0.0  1000000.0  2.039366e-08 -4.900146e-07 -0.041618
['airfoil', 'alpha_deg', 'Re_phys', 'CL', 'CD', 'LD']
     airfoil  alpha_deg    Re_phys            CL            CD        LD
0  eppler420        0.0  1000000.0 -7.537163e-08 -3.262230e-07  0.231043
1  eppler420        0.0  2000000.0 -8.534924e-08 -3.494857e-07  0.244214
2  eppler420        4.0  1000000.0 -4.650078e-08 -3.172909e-07  0.146556
3  eppler420        4.0  2000000.0 -5.482979e-08 -3.410014e-07  0.160791
4   naca0010        0.0  1000000.0  1.012056e-06 -2.361087e-06 -0.428640
['airfoil', 'alpha_deg', 'Re_phys', 'CL', 'CD', 'LD']


In [32]:
print(df_val.describe(include="all"))
print(df_test.describe(include="all"))

         airfoil  alpha_deg       Re_phys            CL            CD  \
count         16  16.000000  1.600000e+01  1.600000e+01  1.600000e+01   
unique         4        NaN           NaN           NaN           NaN   
top     naca0011        NaN           NaN           NaN           NaN   
freq           4        NaN           NaN           NaN           NaN   
mean         NaN   2.000000  1.500000e+06  2.934271e-07 -9.612187e-07   
std          NaN   2.065591  5.163978e+05  5.997400e-07  1.111462e-06   
min          NaN   0.000000  1.000000e+06 -1.352327e-07 -3.042655e-06   
25%          NaN   0.000000  1.000000e+06 -6.714357e-08 -1.046578e-06   
50%          NaN   2.000000  1.500000e+06 -4.088265e-09 -3.945796e-07   
75%          NaN   4.000000  2.000000e+06  3.345848e-07 -2.721555e-07   
max          NaN   4.000000  2.000000e+06  1.429059e-06 -2.207799e-07   

               LD  
count   16.000000  
unique        NaN  
top           NaN  
freq          NaN  
mean     0.021496  
std

kkkk

In [33]:
# ============================================================================
# CELL 7A ─ Reload patched postprocess helpers
# ============================================================================

import importlib
import postprocess
importlib.reload(postprocess)

from postprocess import (
    validate_model_on_library,
    export_run_report,
)

In [35]:
import importlib
import losses
import train_loop

importlib.reload(losses)
importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [36]:
# ============================================================================
# CELL 7B ─ Validation + reports
# ============================================================================

from pathlib import Path
import pandas as pd

if STAGE in (1, 2):
    VAL_STAGE_DIR = Path(VAL_OUT_DIR) / f"stage_{STAGE}"
    VAL_STAGE_DIR.mkdir(parents=True, exist_ok=True)

    all_reports = {}

    # ------------------------------------------------------------
    # Validation library
    # ------------------------------------------------------------
    if len(VAL_AIRFOILS) > 0:
        df_val, val_summary = validate_model_on_library(
            model=model,
            airfoil_library=VAL_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir=VAL_STAGE_DIR / "validation",
            device=device,
            max_cases=24,
        )
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)
        print(df_val.head())
    else:
        df_val = None
        val_summary = {"message": "No validation airfoils available"}
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)

    # ------------------------------------------------------------
    # Test library
    # ------------------------------------------------------------
    if "TEST_AIRFOILS" in globals() and len(TEST_AIRFOILS) > 0:
        df_test, test_summary = validate_model_on_library(
            model=model,
            airfoil_library=TEST_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir=VAL_STAGE_DIR / "test",
            device=device,
            max_cases=24,
        )
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)
        print(df_test.head())
    else:
        df_test = None
        test_summary = {"message": "No test airfoils available"}
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)

    # ------------------------------------------------------------
    # Export run report
    # ------------------------------------------------------------
    run_report = export_run_report(
        history=history if "history" in globals() else [],
        extra={
            "stage": STAGE,
            "aoa_list": list(AOA_LIST),
            "re_list": list(RE_LIST),
            "validation_summary": val_summary,
            "test_summary": test_summary,
            "n_train_airfoils": len(TRAIN_AIRFOILS) if "TRAIN_AIRFOILS" in globals() else 0,
            "n_val_airfoils": len(VAL_AIRFOILS) if "VAL_AIRFOILS" in globals() else 0,
            "n_test_airfoils": len(TEST_AIRFOILS) if "TEST_AIRFOILS" in globals() else 0,
        },
        out_dir=Path(OUT_DIR) / "run_report",
    )

    print("Run report:", run_report)

    # ------------------------------------------------------------
    # Quick sanity print
    # ------------------------------------------------------------
    if df_val is not None and len(df_val) > 0:
        print("\nValidation ranges:")
        print("CL:", df_val["CL"].min(), "to", df_val["CL"].max())
        print("CD:", df_val["CD"].min(), "to", df_val["CD"].max())
        print("LD:", df_val["LD"].min(), "to", df_val["LD"].max())

    if df_test is not None and len(df_test) > 0:
        print("\nTest ranges:")
        print("CL:", df_test["CL"].min(), "to", df_test["CL"].max())
        print("CD:", df_test["CD"].min(), "to", df_test["CD"].max())
        print("LD:", df_test["LD"].min(), "to", df_test["LD"].max())

Validation summary: {'n_cases': 16, 'mean_CL': 2.9342714557789365e-07, 'mean_CD': -9.612186903709238e-07, 'mean_LD': 0.02149631219953417, 'csv_path': '/content/drive/MyDrive/AeroPINN/validation/stage_2/validation/validation_metrics.csv'}
    airfoil  alpha_deg    Re_phys            CL            CD        LD
0  naca0011        0.0  1000000.0  1.206572e-06 -2.779071e-06 -0.434164
1  naca0011        0.0  2000000.0  1.139469e-06 -3.042655e-06 -0.374498
2  naca0011        4.0  1000000.0  1.429059e-06 -2.580095e-06 -0.553878
3  naca0011        4.0  2000000.0  1.381140e-06 -2.844046e-06 -0.485625
4  naca2414        0.0  1000000.0  2.039366e-08 -4.900146e-07 -0.041618
Test summary: {'n_cases': 16, 'mean_CL': 1.7729822639508084e-07, 'mean_CD': -8.480651003821228e-07, 'mean_LD': 0.1439771410554338, 'csv_path': '/content/drive/MyDrive/AeroPINN/validation/stage_2/test/validation_metrics.csv'}
     airfoil  alpha_deg    Re_phys            CL            CD        LD
0  eppler420        0.0  1000000

In [ ]:
# # ============================================================================
# # CELL 8 ─ Stage 3 final optimization + export + XAI
# # ============================================================================
# from pathlib import Path
# import gc
# import torch

# if STAGE == 3:
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#     model.eval()

#     # ------------------------------------------------------------
#     # Choose one target airfoil
#     # ------------------------------------------------------------
#     TARGET_AIRFOIL_NAME = "naca2412"

#     # High-Re application defaults
#     OPT_ALPHA = 4.0
#     OPT_RE = 1.0e6
#     OPT_ITERS = 30
#     OPT_MODE = "lbfgs"

#     all_airfoils = []
#     if "TRAIN_AIRFOILS" in globals():
#         all_airfoils += TRAIN_AIRFOILS
#     if "VAL_AIRFOILS" in globals():
#         all_airfoils += VAL_AIRFOILS
#     if "TEST_AIRFOILS" in globals():
#         all_airfoils += TEST_AIRFOILS

#     matches = [a for a in all_airfoils if a["name"] == TARGET_AIRFOIL_NAME]
#     if not matches:
#         raise ValueError(f"Airfoil '{TARGET_AIRFOIL_NAME}' not found in loaded libraries")

#     target = matches[0]

#     if target["parsec_params"] is None:
#         raise ValueError(
#             f"Airfoil '{TARGET_AIRFOIL_NAME}' does not have acceptable PARSEC fit, "
#             f"so Stage 3 optimization cannot proceed safely."
#         )

#     # ------------------------------------------------------------
#     # Output folder for this optimization case
#     # ------------------------------------------------------------
#     case_dir = Path(OPT_OUT_DIR) / f"{TARGET_AIRFOIL_NAME}_a{int(OPT_ALPHA)}_Re{int(OPT_RE)}"
#     case_dir.mkdir(parents=True, exist_ok=True)

#     print("=" * 70)
#     print("STAGE 3 ─ FINAL OPTIMIZATION")
#     print(f"Target airfoil : {TARGET_AIRFOIL_NAME}")
#     print(f"AoA            : {OPT_ALPHA} deg")
#     print(f"Re             : {OPT_RE:.3e}")
#     print(f"Mode           : {OPT_MODE}")
#     print(f"Iterations     : {OPT_ITERS}")
#     print("=" * 70)

#     # ------------------------------------------------------------
#     # Optimize
#     # ------------------------------------------------------------
#     best_params, best_info = optimize_shape(
#         model=model,
#         init_parsec=target["parsec_params"],
#         mode=OPT_MODE,
#         iters=OPT_ITERS,
#         alpha_deg=OPT_ALPHA,
#         Re_phys=OPT_RE,
#         xlim=XLIM,
#         ylim=YLIM,
#         N_int=N_INT,
#         N_near=N_NEAR,
#         N_airfoil=N_AIRFOIL,
#         N_inlet=N_INLET,
#         N_outlet=N_OUTLET,
#         N_top=N_TOP,
#         N_bot=N_BOT,
#         N_wake=N_WAKE,
#         device=device,
#     )

#     # ------------------------------------------------------------
#     # Export optimized airfoil package
#     # ------------------------------------------------------------
#     exported = export_optimized_airfoil(
#         original_geometry=target["geometry"],
#         fitted_parsec_params=target["parsec_params"],
#         optimized_parsec_params=best_params,
#         best_info=best_info,
#         out_dir=case_dir,
#     )
#     print("Optimized export:", exported)

#     # ------------------------------------------------------------
#     # XAI
#     # ------------------------------------------------------------
#     xai_dir = case_dir / "xai"
#     xai_dir.mkdir(parents=True, exist_ok=True)

#     xai_report = run_xai_suite(
#         model=model,
#         geometry=target["geometry"],
#         parsec_params=best_params,
#         alpha_deg=OPT_ALPHA,
#         Re_phys=OPT_RE,
#         xlim=XLIM,
#         ylim=YLIM,
#         loss_weights=adaptive_weights.all_weights() if "adaptive_weights" in globals() and adaptive_weights is not None else None,
#         out_dir=xai_dir,
#         device=device,
#     )

#     print("XAI report saved:", xai_dir / "xai_report.json")
#     print("XAI summary:", xai_report)

#     # ------------------------------------------------------------
#     # Final summary print
#     # ------------------------------------------------------------
#     print("\nStage 3 complete.")
#     print("Saved in:", case_dir)